# AI-Powered Intelligent Sales Agent (Indus University FYP)
## Fine-Tuning TinyLlama 1.1B on Solar Sales Dataset using LoRA / QLoRA
**Authors:** Mujtaba Khan, Israr Khan, Arham ul haq, Aaniq joseph  
**Supervisors:** Sir Dayanand (Lecturer), Engr. Emaan (Lab Engineer)  
**Target Hardware:** Google Colab Free T4 GPU (15GB VRAM)

In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers peft trl bitsandbytes accelerate datasets evaluate

In [ ]:
# Step 2: Verify GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Load Base TinyLlama 1.1B Model with 4-bit NormalFloat Quantization (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print("TinyLlama 1.1B loaded successfully in 4-bit!")

In [ ]:
# Step 4: Configure LoRA Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Load and Format the Solar Sales Dataset
from datasets import load_dataset

# Format into TinyLlama Chat template
def format_example(ex):
    return {
        "text": f"<|system|>\n{ex['instruction']}</s>\n<|user|>\n{ex['input']}</s>\n<|assistant|>\n{ex['output']}</s>"
    }

dataset = load_dataset("json", data_files="solar_sales_train.json")
train_data = dataset["train"].map(format_example)
print(f"Formatted {len(train_data)} training dialogues.")

In [ ]:
# Step 6: Train with SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tinyllama-solar-sales-agent",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting LoRA Fine-Tuning...")
trainer.train()

# Save trained LoRA weights
trainer.model.save_pretrained("./tinyllama-solar-sales-agent")
tokenizer.save_pretrained("./tinyllama-solar-sales-agent")
print("Fine-tuning complete! LoRA weights saved to ./tinyllama-solar-sales-agent")

In [ ]:
# Step 7: Test Inference on Unseen Solar Sales Question
query = "My monthly bill is 45,000 PKR. What solar package do you recommend?"
prompt = f"<|system|>\nYou are an AI Sales Agent for a solar energy company.</s>\n<|user|>\n{query}</s>\n<|assistant|>\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id
)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"Customer Query: {query}\n")
print(f"Fine-Tuned TinyLlama Response:\n{response.strip()}")